# Underwater detection: Kaggle training (swimming / struggling / drowning)
Settings → Accelerator **GPU T4 x2 or P100**, Internet **on**, then Run All. The notebook clones the training pipeline from GitHub and downloads the Figshare *Underwater Drowning Detection Dataset* (CC BY 4.0, 5,613 images, 3 classes, ~1.6 GB) into `/tmp`, so the dataset is never saved as notebook output. Reported 97.5% mAP is reference only.

In [ ]:
from pathlib import Path
import subprocess, sys, shutil, json, urllib.request, zipfile
REPO = 'https://github.com/VuongQuocHung/drowining-detect.git'
DATASET_URL = 'https://ndownloader.figshare.com/files/56039351'  # datasets.zip
SOURCE = Path('/tmp/drowining-detect')
CODE = SOURCE/'model_training/pipeline'
DATASET = Path('/tmp/underwater/datasets')
GROUPS = None  # Optional CSV columns: image,group; image relative to images/.
PREPARED = Path('/kaggle/working/prepared')
OUTPUT = Path('/kaggle/working/experiments')
if not (CODE/'audit.py').exists():
    # Sparse checkout: only the pipeline, not model weights or demo videos.
    subprocess.run(['git','clone','--depth','1','--filter=blob:none','--sparse',REPO,str(SOURCE)],check=True)
    subprocess.run(['git','-C',str(SOURCE),'sparse-checkout','set','model_training/pipeline'],check=True)
if not (DATASET/'images/train').exists():
    archive = Path('/tmp/datasets.zip')
    if not archive.exists():
        def progress(blocks, size, total):
            done = blocks*size
            if blocks % 2000 == 0 or done >= total:
                print(f'\rDownloading dataset: {min(done,total)/1e6:,.0f} / {total/1e6:,.0f} MB', end='', flush=True)
        urllib.request.urlretrieve(DATASET_URL, archive, progress)
        print()
    with zipfile.ZipFile(archive) as zipped:
        zipped.extractall(DATASET.parent)
    archive.unlink()
assert (CODE/'audit.py').exists(), 'Pipeline clone failed'
assert (DATASET/'images/train').exists(), 'Unexpected dataset layout'
print('Train images:', len(list((DATASET/'images/train').iterdir())), '| Val images:', len(list((DATASET/'images/val').iterdir())))
print('Classes:', (DATASET/'labels/train/classes.txt').read_text().split())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE/'requirements-kaggle.txt')], check=True)

Restart the Python session after installation if Torch/NumPy were already imported, then rerun path definitions. Internet is needed for dependencies/pretrained weights. Save a notebook version with outputs to retain checkpoints.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable GPU accelerator'
import torchvision
print('Python:', sys.version)
print('Torch:', torch.__version__, 'TorchVision:', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0))
boxes = torch.tensor([[0., 0., 1., 1.]], device='cuda')
scores = torch.tensor([1.], device='cuda')
assert torchvision.ops.nms(boxes, scores, 0.5).numel() == 1
command=[sys.executable,str(CODE/'audit.py'),str(DATASET),'--output',str(PREPARED)]
if GROUPS: command += ['--groups',str(GROUPS)]
subprocess.run(command,check=True)


In [ ]:
from IPython.display import display, Image
report=json.loads((PREPARED/'audit.json').read_text())
print(json.dumps({k:v for k,v in report.items() if k not in ('near_cross_split','exact_cross_split')},indent=2))
print('Near-duplicate cross-split pairs:',len(report['near_cross_split']))
display(Image(filename=str(PREPARED/'annotations.jpg')))


Review annotation previews and audit.json before training. Exact cross-split duplicates/invalid labels stop preparation. With no source groups, retain original splits and report their limitations; do not present validation as independent test data.

In [ ]:
# Resume: attach a previous saved notebook output, then copy experiments/ to OUTPUT.
# shutil.copytree('/kaggle/input/previous-run/experiments',OUTPUT,dirs_exist_ok=True)
subprocess.run([sys.executable,str(CODE/'train.py'),'--data',str(PREPARED/'data.yaml'),
                '--output',str(OUTPUT),'--resume'],check=True)


In [ ]:
comparison=json.loads((OUTPUT/'comparison.json').read_text())
selected=json.loads((OUTPUT/'selected.json').read_text())
print(json.dumps(comparison,indent=2))
print('Selected by validation:',selected['checkpoint'])


## Held-out test
Run only after freezing selection. Skip if data.yaml has no independent test split. Test scores never choose weights.

In [ ]:
RUN_HELD_OUT_TEST=False
if RUN_HELD_OUT_TEST:
    subprocess.run([sys.executable,str(CODE/'train.py'),'--data',str(PREPARED/'data.yaml'),
                    '--output',str(OUTPUT),'--test-checkpoint',selected['checkpoint']],check=True)


## Deploy and rollback
Download `underwater_results.zip` from the Output panel (below). It contains `underwater_selected.pt`, selected/comparison/environment JSON, audit/manifest and plots; intermediate epoch checkpoints are left out to keep it small. On the backend machine set `UNDERWATER_MODEL` to the new checkpoint; keep `underwaterModel.pt` untouched for rollback. Evaluate independent labeled videos with evaluate_events.py. No accuracy or soak result is claimed until an actual run completes.

In [ ]:
# Bundle the deliverables for one-click download (epoch checkpoints excluded).
bundle = Path('/kaggle/working/underwater_results.zip')
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as zipped:
    for file in sorted(OUTPUT.rglob('*')):
        if file.is_file() and not file.name.startswith('epoch'):
            zipped.write(file, file.relative_to(OUTPUT.parent))
print(f'{bundle} ({bundle.stat().st_size/1e6:.1f} MB)')